# Medical Calculators



Read `PROJECT.pdf`. Student TODOs are here; helpers supply the mechanics. Group presentation and individual reflection remain the final deliverables.

In [ ]:
# Upload the student ZIP in Colab, or run from the unzipped project locally.
from pathlib import Path
import os, zipfile
if not Path('data/questions.jsonl').exists():
    from google.colab import files
    uploaded=files.upload()
    archives=[name for name in uploaded if name.endswith('.zip')]
    if len(archives)!=1:
        raise ValueError('Upload exactly one student ZIP.')
    root=Path('/content').resolve()
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            if not (root/member.filename).resolve().is_relative_to(root):
                raise ValueError('Unsafe ZIP path')
        z.extractall(root)
    os.chdir(root/'medcalc_proj')
print('Project folder:',Path.cwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json, importlib, pandas as pd
from helpers.experiment import load_questions, run_experiment
from helpers.prompts import direct_solver, diagnostic_solver, task_input
from helpers.evaluation import baseline_score, score_run, summary, POLICY

# Optional: set OUTPUT to a mounted Drive folder to retain results across Colab sessions.
OUTPUT=Path('outputs')
OUTPUT.mkdir(parents=True,exist_ok=True)
dev_rows=load_questions(split='dev')
test_rows=load_questions(split='test')
examples=json.loads(Path('data/examples.json').read_text())
print('Development:',len(dev_rows),'Test:',len(test_rows),'Extra worked examples:',len(examples))
display(pd.read_csv('data/calculator_summary.csv'))
print('Solver input fields:',list(task_input(dev_rows[0])))
from helpers.references import load_reference_library, calculator_reference, render_reference
from IPython.display import Markdown
references=load_reference_library()
print('Original policy:',POLICY)

## Warm-up: what does the original evaluation measure?

For numerical calculators the final number must lie inside inclusive supplied bounds, generally **±5%**. Rule scores use Python `round` (ties to even), not always rounding upward. Units and intermediate reasoning are ignored. JSON parsing/inference failures count as incorrect. The following examples expose these decisions before any model runs.

In [ ]:
numerical=next(r for r in dev_rows if r['kind']=='numerical')
rule=next(r for r in dev_rows if r['kind']=='rule')
print('Numerical reference/bounds:',numerical['gold_answer'],numerical['lower'],numerical['upper'])
for value in [numerical['lower'],numerical['gold_answer'],numerical['upper'],numerical['upper']+1]:
    print(value,baseline_score({'answer':str(value),'unit':'deliberately wrong unit'},numerical))
print('Python ties:',[(x,round(x)) for x in [2.4,2.5,2.6,3.5]])
print('Invalid numeric JSON:',baseline_score('not JSON',rule))

## Starting solving model

Choose a model from `docs/MODEL_GUIDE.md`; default Qwen2.5-3B in 4-bit mode. Keep it fixed for your main solving comparisons. Non-Qwen choices are available. No paid API is required. The output budget starts at 1024 tokens; inspect and retain truncated/failed outputs.

In [ ]:
from helpers.models import HFModel, MODEL_OPTIONS
MODEL_KEY='qwen_default'
model=HFModel(MODEL_OPTIONS[MODEL_KEY],four_bit=True,max_new_tokens=1024,temperature=0.0)
print(model.config)
(OUTPUT/'model_config.json').write_text(json.dumps(model.config,indent=2))
import subprocess,sys
(OUTPUT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
smoke=run_experiment(dev_rows[:1],model,direct_solver,'direct_smoke',OUTPUT)
print(smoke[0]['raw'])
print('Inference error, if any:',smoke[0]['error'])

In [ ]:
direct_results=run_experiment(dev_rows,model,direct_solver,'direct',OUTPUT)
direct_scores=score_run(direct_results,dev_rows)
display(summary(direct_scores))

## Human evaluation of direct answers

Start by calculating the answers yourself, then check whether you agree with the current direct-answer evaluation.

1. **Do the calculation independently.** Read the patient note and question. Find the equation, scoring rules and units in **`docs/CALCULATOR_REFERENCE.md`**, the same cards are in `data/calculator_references.json`. Record the inputs you use, your calculation, final answer and unit before looking at the model's answer, the patient-specific reference answer or the evaluator's label. An independent calculation worksheet is exported below.
2. **Check the model's direct answer.** Compare it with your calculation and decide whether you would accept it. Record your own judgment and explain any unit, precision or note/reference issue that affected it.
3. **Compare with the current evaluation result.** Once your independent judgment is recorded, check whether the original final-answer metric labels the same response as correct or incorrect. Do you agree? Explain differences caused by tolerance, rounding, units, parsing or a disputed reference answer. Use development cases for these policy discussions; defer reserved-case comparisons until final judge validation.
4. **Discuss agreement with your group.** Compare independently recorded judgments, investigate disagreements and refine your evaluation rubric with evidence. Keep your original calculations and labels.

Suggested starter: 24 shared direct responses per reviewer, with 16 development questions and eight reserved for later judge validation. Keep all variants of a question in its original partition. `docs/ANNOTATION_GUIDE.md` explains the forms and labels.

A direct-only response does not display its inputs, formula or arithmetic. You can check its final answer against your own calculation, but its internal calculation process remains unobservable.

In [ ]:
from helpers.annotations import (select_review,make_review_files,read_annotations,
    pairwise_agreement,load_adjudication,compare_baseline,ASPECTS)
ANNOTATORS=['A','B','C']  # TODO: each group member's unique short ID.
REVIEW_N=24
JUDGE_VALIDATION_N=8
DIRECT_REVIEW_DIR=OUTPUT/'direct_human_review_v1'
direct_review_outputs=select_review(direct_results,dev_rows,n=REVIEW_N)
if not (DIRECT_REVIEW_DIR/'review_key.json').exists():
    make_review_files(direct_review_outputs,dev_rows,DIRECT_REVIEW_DIR,ANNOTATORS,
                     JUDGE_VALIDATION_N,prefix='direct')
direct_review_key=json.loads((DIRECT_REVIEW_DIR/'review_key.json').read_text())
question_partition={r['question_id']:r['partition'] for r in direct_review_key}
print('Direct review forms:',[str(DIRECT_REVIEW_DIR/f'annotations_{a}.csv') for a in ANNOTATORS])
print('Review the note, candidate and references; do not open individual baseline votes yet.')
# First-stage worksheets contain the note/question/formula card, with no model or patient-gold answer.
calculation_questions={r['id']:r for r in dev_rows}
calculation_rows=[]
for record in direct_review_key:
    question=calculation_questions[record['question_id']]
    calculation_rows.append({'response_id':record['response_id'], 'question_id':question['id'],
        'note':question['note'],'question':question['question'],
        'calculator_reference':render_reference(calculator_reference(question['calculator_id'],references)),
        'your_inputs':'','your_calculation':'','your_answer':'','your_unit':'','assumptions':''})
for annotator in ANNOTATORS:
    worksheet=DIRECT_REVIEW_DIR/f'independent_calculations_{annotator}.csv'
    if not worksheet.exists():pd.DataFrame(calculation_rows).to_csv(worksheet,index=False)
print('START HERE:',[str(DIRECT_REVIEW_DIR/f'independent_calculations_{a}.csv') for a in ANNOTATORS])
print('After recording calculations, open your annotations CSV to assess the direct answers.')

### TODO — record your calculations and human judgments

Complete your independent calculation worksheet first. Then judge the model's direct answers and record your labels before checking the baseline votes. Explain where your decisions and the original metric agree or differ. Use these observations to define a preliminary evaluation rubric for answer validity, units and precision; record its version.

Do not copy the supplied patient-specific reference answer as your own calculation. Record disputed reference assumptions. For direct-only outputs, keep unobservable process dimensions separate from final-answer validity.

## After doing the calculations: does the current tolerance make sense?

Use your independent calculations and the model's direct outputs to examine the original numerical bounds, generally **±5%**, and rule-score rounding. Are these policies appropriate for these calculators? Which disagreements are explained by the tolerance? Could a stricter policy reject a valid rounded result, or could a loose policy accept a meaningfully wrong answer?

**TODO:** propose and implement your own final-answer scoring policy. You may change the relative/absolute numerical tolerance, use calculator-specific precision, or alter how rule scores are accepted. Justify the choice using the calculator card and human calculations rather than choosing a threshold merely to obtain a preferred score. Decide how units are handled. A tighter numerical threshold alone does not verify the model's reasoning.

The cell below gives an editable starting point. Keep the original evaluator unchanged and re-score the same saved outputs with your new policy. Compare changed labels and accuracy with your human judgments. Develop the policy on development cases and carry the settled rule into your later human/LLM judging rubric; do not tune it on reserved validation cases.

In [ ]:
from helpers.evaluation import numeric_answer

RELATIVE_TOLERANCE=None  # TODO: a nonnegative fraction (e.g. 0.01 means 1%); choose and justify.
ABSOLUTE_TOLERANCE=None  # TODO: nonnegative allowance in the expected output unit; choose and justify.
RULE_POLICY=None  # TODO: 'exact_integer' or 'baseline_round', or implement another justified rule.

def revised_final_answer_score(raw,gold):
    value=numeric_answer(raw)
    if value is None:return False  # Keep parsing/inference failures in the denominator.
    if gold['kind']=='rule':
        if RULE_POLICY=='baseline_round':return round(value)==gold['gold_answer']
        if RULE_POLICY=='exact_integer':return value.is_integer() and value==gold['gold_answer']
        raise NotImplementedError('Choose or implement your rule-score policy.')
    if RELATIVE_TOLERANCE is None or ABSOLUTE_TOLERANCE is None:
        raise NotImplementedError('Choose numerical tolerances first.')
    if RELATIVE_TOLERANCE<0 or ABSOLUTE_TOLERANCE<0:raise ValueError('Tolerances must be nonnegative.')
    # Editable example: max(relative allowance, absolute allowance).
    # Replace this with a calculator-specific precision rule if appropriate.
    allowed=max(RELATIVE_TOLERANCE*abs(gold['gold_answer']),ABSOLUTE_TOLERANCE)
    # TODO: decide whether/how to check or convert units; this starter compares numbers only.
    return abs(value-gold['gold_answer'])<=allowed

RUN_REVISED_SCORING=False  # TODO: enable after choosing/implementing the policy.
if RUN_REVISED_SCORING:
    calibration_question_ids={r['question_id'] for r in direct_review_key if r['partition']=='judge_calibration'}
    gold_by_id={r['id']:r for r in dev_rows}
    revised_rows=[]
    for output in direct_results:
        if output['id'] not in calibration_question_ids:continue
        gold=gold_by_id[output['id']]
        revised_rows.append({'id':output['id'],'kind':gold['kind'],
            'original_correct':baseline_score(output['raw'],gold)['answer_correct'],
            'revised_correct':revised_final_answer_score(output['raw'],gold)})
    revised_comparison=pd.DataFrame(revised_rows)
    display(revised_comparison[revised_comparison.original_correct!=revised_comparison.revised_correct])
    print('Original and revised accuracy on the same development responses:')
    print(revised_comparison[['original_correct','revised_correct']].mean())
    revised_comparison.to_csv(OUTPUT/'revised_tolerance_comparison.csv',index=False)
    # Join your independent/adjudicated human labels using review_key's response_id/question_id mapping.
    # TODO: explain changed decisions using independent calculations and reference precision.


## Agreement and the original evaluation

Report pairwise agreement and kappa on the initial direct-answer labels. Constant unobservable labels may have undefined kappa; this does not establish reasoning correctness. Discuss final-answer disagreements, document adjudication with evidence, and preserve original labels. Use development questions for rubric/judge changes; keep reserved questions out of tuning.

Compare human and baseline final-answer judgments on the same direct output. What differences arise from units, rounding, parsing or tolerance? What questions remain unanswered because the model did not show its calculation?

In [ ]:
DIRECT_HUMAN_READY=False  # TODO: after independent direct-review CSVs are complete.
DIRECT_ADJUDICATION_READY=False  # TODO: after group discussion and documented adjudication.
if DIRECT_HUMAN_READY:
    direct_humans=read_annotations([DIRECT_REVIEW_DIR/f'annotations_{a}.csv' for a in ANNOTATORS])
    direct_cal_ids={r['response_id'] for r in direct_review_key if r['partition']=='judge_calibration'}
    direct_agreement=pairwise_agreement(direct_humans[direct_humans.response_id.isin(direct_cal_ids)])
    display(direct_agreement)
    direct_agreement.to_json(OUTPUT/'direct_human_agreement.json',orient='records',indent=2)
    # Compare each recorded independent judgment with the current metric before adjudication.
    calibration_key=[r for r in direct_review_key if r['response_id'] in direct_cal_ids]
    for annotator in ANNOTATORS:
        personal=direct_humans[(direct_humans.annotator==annotator)&direct_humans.response_id.isin(direct_cal_ids)]
        personal_report,personal_cases=compare_baseline(calibration_key,personal)
        print('Independent human vs current metric:',annotator,personal_report)
        display(personal_cases[['response_id','answer','baseline_answer','answer_disagreement']])
        personal_cases.to_csv(OUTPUT/f'independent_human_vs_metric_{annotator}.csv',index=False)
    # TODO: explain agreement/disagreement using your saved calculation evidence.

if DIRECT_ADJUDICATION_READY:
    if not DIRECT_HUMAN_READY:raise ValueError('Complete independent review first.')
    direct_adjudicated=load_adjudication(DIRECT_REVIEW_DIR/'adjudication.csv',[r['response_id'] for r in direct_review_key])
    direct_metric_report,direct_metric_cases=compare_baseline(direct_review_key,direct_adjudicated)
    display(direct_metric_cases[direct_metric_cases.response_id.isin(direct_cal_ids)])
    direct_metric_cases.to_csv(OUTPUT/'direct_human_vs_metric.csv',index=False)

### TODO — what can your current evaluation establish?

Report human agreement and explain adjudication decisions. Why do human and original metric judgments differ? Why do judgments differ between human reviewers? Propose changes to the final-answer policy. Could either evaluation tell whether an apparently correct number came from a valid calculation? Explain why an absent worked process should remain unobservable.

## Reveal the reasoning, try CoT first

Your rubric can judge the final answer, but a direct response gives little evidence about how the model applied the calculator. Could a prompt requesting a worked solution make formula choice, input extraction, units and arithmetic assessable?

Consult `docs/METHODS.md`. **TODO: first implement zero-shot CoT** by writing an instruction for a concise, checkable calculation before the answer. `cot_messages` supplies the output schema and original task; you decide the instruction. Print the prompt, smoke test one case, then run development questions. A displayed solution is evidence to inspect, not a guaranteed faithful account of hidden reasoning or proof of correctness.

In [ ]:
from helpers.guided_methods import cot_messages,one_shot_messages,grounded_messages,refine_once
COT_INSTRUCTION=None  # TODO: request supported inputs, formula/rules, units and a worked calculation.
def solve_cot(task,model):
    if not COT_INSTRUCTION:raise NotImplementedError('First write COT_INSTRUCTION.')
    return model.generate(cot_messages(task,COT_INSTRUCTION))
RUN_COT=False  # Set True after implementing the instruction.
CASE_INDEX=0
case=dev_rows[CASE_INDEX]
if RUN_COT:
    print(cot_messages(task_input(case),COT_INSTRUCTION))
    cot_smoke=run_experiment([case],model,solve_cot,'cot_smoke',OUTPUT,tag=COT_INSTRUCTION)
    print(cot_smoke[0]['raw'])
    cot_results=run_experiment(dev_rows,model,solve_cot,'cot',OUTPUT,tag=COT_INSTRUCTION)
    display(summary(score_run(cot_results,dev_rows)))
    display(Markdown(render_reference(calculator_reference(case['calculator_id'],references))))

## Then try one-shot CoT

**TODO: next implement one-shot CoT.** The examples are in **`data/examples.json`**, loaded as `examples` near the start of the notebook. Keys are calculator IDs as strings; for a task use `examples[str(task['calculator_id'])]`. Each entry has a separate note, question, reference entities and worked explanation. Those examples do not overlap evaluation notes.

Inspect the example and its formula assumptions. `one_shot_messages` inserts one same-calculator user/assistant demonstration before the new task. Write an instruction that transfers the method while extracting the new patient's facts independently. Never insert the evaluated patient's gold fields. Keep the worked output schema consistent with zero-shot CoT.

In [ ]:
ONE_SHOT_INSTRUCTION=None  # TODO: your one-shot CoT instruction.
example=examples[str(case['calculator_id'])]
print('Example source: data/examples.json; calculator:',example['calculator'])
print('Example question:',example['question'])
print('Worked demonstration:',example['gold_explanation'])
def solve_one_shot(task,model):
    if not ONE_SHOT_INSTRUCTION:raise NotImplementedError('Write ONE_SHOT_INSTRUCTION after CoT.')
    return model.generate(one_shot_messages(task,examples,ONE_SHOT_INSTRUCTION))
RUN_ONE_SHOT=False
if RUN_ONE_SHOT:
    if not RUN_COT:raise ValueError('Run the zero-shot CoT comparison first.')
    print(one_shot_messages(task_input(case),examples,ONE_SHOT_INSTRUCTION))
    one_shot_smoke=run_experiment([case],model,solve_one_shot,'one_shot_smoke',OUTPUT,tag=ONE_SHOT_INSTRUCTION)
    print(one_shot_smoke[0]['raw'])
    one_shot_results=run_experiment(dev_rows,model,solve_one_shot,'one_shot',OUTPUT,tag=ONE_SHOT_INSTRUCTION)
    display(summary(pd.concat([direct_scores,score_run(cot_results,dev_rows),
                              score_run(one_shot_results,dev_rows)],ignore_index=True)))

## Does the displayed reasoning satisfy your rubric?

Apply your existing human rubric to the newly visible steps, refining process definitions on development cases if necessary. Inspect the note and formula card: are inputs/time-points supported, thresholds and units valid, and arithmetic correct? Look for correct-number/wrong-process cases and valid equivalent formulas. Record the added process criteria and rubric version. If you change answer criteria, document reannotation rather than mixing incompatible labels.

The starter selects four development and four reserved questions from the initial review, then exports **both CoT and one-shot CoT outputs**: 16 additional responses per member. These question partitions stay fixed. You may reduce or expand this supplement for available time and report its coverage. No need to label every output of every method.

In [ ]:
REASONING_REVIEW_DIR=OUTPUT/'reasoning_human_review_v1'
TRACE_QUESTIONS_PER_PARTITION=4  # Adjustable: 8 questions x 2 methods = 16 supplementary responses.
if RUN_COT and RUN_ONE_SHOT:
    trace_question_ids=[]
    for part in ['judge_calibration','judge_validation']:
        available=[r['question_id'] for r in direct_review_key if r['partition']==part]
        if TRACE_QUESTIONS_PER_PARTITION>len(available):raise ValueError('Requested more questions than this partition has.')
        trace_question_ids.extend(available[:TRACE_QUESTIONS_PER_PARTITION])
    reasoning_review_outputs=[r for rows in [cot_results,one_shot_results] for r in rows if r['id'] in trace_question_ids]
    if not (REASONING_REVIEW_DIR/'review_key.json').exists():
        make_review_files(reasoning_review_outputs,dev_rows,REASONING_REVIEW_DIR,ANNOTATORS,
                         partitions=question_partition,prefix='trace')
    reasoning_review_key=json.loads((REASONING_REVIEW_DIR/'review_key.json').read_text())
    print('Reasoning review forms:',[str(REASONING_REVIEW_DIR/f'annotations_{a}.csv') for a in ANNOTATORS])
    print('Additional shared responses:',len(reasoning_review_key))

REASONING_HUMAN_READY=False  # TODO: after independent supplementary review.
REASONING_ADJUDICATION_READY=False
if REASONING_HUMAN_READY:
    reasoning_humans=read_annotations([REASONING_REVIEW_DIR/f'annotations_{a}.csv' for a in ANNOTATORS])
    trace_cal_ids={r['response_id'] for r in reasoning_review_key if r['partition']=='judge_calibration'}
    display(pairwise_agreement(reasoning_humans[reasoning_humans.response_id.isin(trace_cal_ids)]))
if REASONING_ADJUDICATION_READY:
    if not REASONING_HUMAN_READY:raise ValueError('Complete independent reasoning review first.')
    reasoning_adjudicated=load_adjudication(REASONING_REVIEW_DIR/'adjudication.csv',[r['response_id'] for r in reasoning_review_key])
    trace_metric_report,trace_metric_cases=compare_baseline(reasoning_review_key,reasoning_adjudicated)
    display(trace_metric_cases[trace_metric_cases.response_id.isin(trace_cal_ids)])
    trace_metric_cases.to_csv(OUTPUT/'reasoning_human_vs_metric.csv',index=False)

### TODO — compare CoT and one-shot CoT reasoning

Do the worked solutions satisfy your rubric? Support the comparison with matched response excerpts and reference checks. Did an example improve reasoning, merely improve formatting, or introduce a copied assumption? Separate final-answer correctness from process validity, and discuss disagreement between human reviewers. Preserve independent labels and adjudication evidence.

## Optional methods from docs/METHODS.md

After CoT and one-shot CoT, you may explore formula grounding, Self-Refine, MedPrompt adaptations or other references in `docs/METHODS.md`. They are optional extensions. The project now moves from human evaluation to scaling your rubric with an LLM judge; no particular additional solving method is required.

In [ ]:
GROUNDED_INSTRUCTION=None  # TODO: your formula/card instruction.

def solve_grounded(task,model):
    if not GROUNDED_INSTRUCTION:raise NotImplementedError('Write GROUNDED_INSTRUCTION.')
    return model.generate(grounded_messages(task,references,GROUNDED_INSTRUCTION))

RUN_GROUNDED=False
if RUN_GROUNDED:
    print(json.dumps(grounded_messages(task_input(case),references,GROUNDED_INSTRUCTION),indent=2))
    grounded_smoke=run_experiment([case],model,solve_grounded,'grounded_smoke',OUTPUT,
                                tag=json.dumps({'grounded':GROUNDED_INSTRUCTION}))
    print(grounded_smoke[0]['raw'])
    grounded_results=run_experiment(dev_rows,model,solve_grounded,'grounded',OUTPUT,
                                   tag=json.dumps({'grounded':GROUNDED_INSTRUCTION}))

In [ ]:
CRITIQUE_INSTRUCTION=None  # Optional TODO: evidence, units, formula/rules and arithmetic checks.
REVISION_INSTRUCTION=None  # Optional TODO: how to act on feedback, including no-error cases.
def solve_refined(task,model):
    if not CRITIQUE_INSTRUCTION or not REVISION_INSTRUCTION:raise NotImplementedError('Write both refinement instructions.')
    draft=solve_cot(task,model)
    return refine_once(task,model,draft,CRITIQUE_INSTRUCTION,REVISION_INSTRUCTION)
RUN_REFINE=False
if RUN_REFINE:
    refined_results=run_experiment(dev_rows,model,solve_refined,'refined',OUTPUT,
        tag=json.dumps({'critique':CRITIQUE_INSTRUCTION,'revision':REVISION_INSTRUCTION}))

method_outputs={'direct':direct_results}
if RUN_COT:method_outputs['cot']=cot_results
if RUN_ONE_SHOT:method_outputs['one_shot']=one_shot_results
if RUN_GROUNDED:method_outputs['grounded']=grounded_results
if RUN_REFINE:method_outputs['refined']=refined_results
display(summary(pd.concat([score_run(rows,dev_rows) for rows in method_outputs.values()],ignore_index=True)))

## Human review takes time: could an LLM apply your rubric?

You have a rubric, independent labels, disagreement evidence and adjudicated decisions. Repeating this review for every model output is time-consuming. Could an LLM judge help apply the same criteria at scale? Treat that as a hypothesis: the judge may miss the same errors as the solver or reject valid solutions.

Build it step by step from **your group's existing rubric**, then verify its decisions against the human reference. Keep separate scores for direct-only and reasoning responses: many unobservable direct processes must not inflate claims about reasoning-error detection.

In [ ]:
from helpers.judging import judge_messages,run_judges,compare_with_humans,error_table
# Prepare the shared human reference only after both review stages are complete.
ADJUDICATION_READY=DIRECT_ADJUDICATION_READY and REASONING_ADJUDICATION_READY
if ADJUDICATION_READY:
    review_key=direct_review_key+reasoning_review_key
    human_labels=pd.concat([direct_humans,reasoning_humans],ignore_index=True)
    adjudicated=pd.concat([direct_adjudicated,reasoning_adjudicated],ignore_index=True)
    if adjudicated.response_id.duplicated().any():raise ValueError('Review IDs must be unique across stages.')
    calibration_ids={r['response_id'] for r in review_key if r['partition']=='judge_calibration'}
    validation_ids={r['response_id'] for r in review_key if r['partition']=='judge_validation'}
    judge_items=[{'response_id':r['response_id'],'question_id':r['question_id'],
                  'candidate_raw':r['output']['raw']} for r in review_key]
    calibration_items=[r for r in judge_items if r['response_id'] in calibration_ids]
    validation_items=[r for r in judge_items if r['response_id'] in validation_ids]
    print('Judge development/validation response counts:',len(calibration_items),len(validation_items))

### Build the judge, step by step

1. Translate your human rubric into explicit aspect criteria and decision rules. Keep the same tolerance and overall-label meanings.
2. Decide what evidence the judge receives: original note, candidate, calculator card and evaluation-only references. Ask it to distinguish local arithmetic validity from an upstream extraction error.
3. Require labels and supporting evidence in a consistent structure. Define which absent calculation steps are unobservable; do not automatically accept a correct final number.
4. Print and test a prompt on a development response. Inspect its labels **and** evidence against human adjudication before running the development batch.

The helper supplies message construction and JSON parsing, not substantive correctness. You can change the scaffold to reflect your group's design.

In [ ]:
JUDGE_CRITERIA=None  # TODO: translate your human aspect definitions into judge instructions.
JUDGE_DECISION_RULES=None  # TODO: tolerance and overall policy; unobservable direct process.
JUDGE_EVIDENCE_RULES=None  # TODO: exact candidate excerpts and reference/calculation checks.
JUDGE_RUBRIC_V1=None
if all([JUDGE_CRITERIA,JUDGE_DECISION_RULES,JUDGE_EVIDENCE_RULES]):
    JUDGE_RUBRIC_V1='\n'.join([JUDGE_CRITERIA,JUDGE_DECISION_RULES,JUDGE_EVIDENCE_RULES])
judge_model=model  # Reuse the small model, or choose another from docs/MODEL_GUIDE.md.
RUN_JUDGE_SMOKE=False
if RUN_JUDGE_SMOKE:
    if not ADJUDICATION_READY or not JUDGE_RUBRIC_V1:raise ValueError('Finish human reviews and judge instructions first.')
    item=next(r for r in calibration_items if r['response_id'].startswith('trace_'))
    gold=next(r for r in dev_rows if r['id']==item['question_id'])
    prompt=judge_messages(item['candidate_raw'],gold,calculator_reference(gold['calculator_id'],references),JUDGE_RUBRIC_V1)
    print(json.dumps(prompt,indent=2))
    judge_smoke=run_judges([item],dev_rows,references,judge_model,JUDGE_RUBRIC_V1,'judge_smoke',OUTPUT/'judges')
    print(judge_smoke[0])
    print('Human reference:',adjudicated[adjudicated.response_id==item['response_id']].to_dict('records'))

In [ ]:
RUN_JUDGE_CALIBRATION=False
if RUN_JUDGE_CALIBRATION:
    if not ADJUDICATION_READY or not JUDGE_RUBRIC_V1:raise ValueError('Finish human reviews and judge instructions first.')
    human_cal=adjudicated[adjudicated.response_id.isin(calibration_ids)]
    judge_v1_cal=run_judges(calibration_items,dev_rows,references,judge_model,JUDGE_RUBRIC_V1,'judge_v1_cal',OUTPUT/'judges')
    display(compare_with_humans(judge_v1_cal,human_cal))
    # TODO: inspect disagreements and supporting evidence on calibration cases only.

### TODO — improve the judge from observed errors

Describe a judging failure on calibration cases and implement one meaningful change. For example: clearer equivalent-formula handling, a policy for missing evidence, adjudicated calibration demonstrations, or deterministic arithmetic checks. Justify your choice, preserve v1, and record what changed. You can change the helper/interface if your design requires it. Do not tune on reserved validation cases.

In [ ]:
JUDGE_RUBRIC_V2=None  # TODO: improved rubric or adapt the runner to your new judge design.
RUN_JUDGE_V2_CALIBRATION=False
if RUN_JUDGE_V2_CALIBRATION:
    if not JUDGE_RUBRIC_V2 or not ADJUDICATION_READY:raise ValueError('Write improved judge design and complete adjudication.')
    human_cal=adjudicated[adjudicated.response_id.isin(calibration_ids)]
    judge_v2_cal=run_judges(calibration_items,dev_rows,references,judge_model,JUDGE_RUBRIC_V2,'judge_v2_cal',OUTPUT/'judges')
    display(compare_with_humans(judge_v2_cal,human_cal))

## Does the evaluator work correctly? Held-out human comparison

Freeze both versions and your rubric before judging reserved responses. Compare versions on exactly the same adjudicated human labels. Report per-dimension agreement, kappa, false alarms, missed errors, unobservable labels and judge failures. Inspect evidence: agreement alone is not correctness. The default eight validation reasoning responses (from four questions) support only a small exploratory study; expand before making broad claims.

In [ ]:
RUN_JUDGE_VALIDATION=False  # TODO: enable only after settling the judge.
if RUN_JUDGE_VALIDATION:
    if not JUDGE_RUBRIC_V1 or not JUDGE_RUBRIC_V2 or not ADJUDICATION_READY:raise ValueError('Settle both judge versions and adjudication first.')
    human_val=adjudicated[adjudicated.response_id.isin(validation_ids)]
    validation_reports=[]
    for name,rubric in [('v1',JUDGE_RUBRIC_V1),('v2',JUDGE_RUBRIC_V2)]:
        records=run_judges(validation_items,dev_rows,references,judge_model,rubric,'judge_'+name+'_validation',OUTPUT/'judges')
        report=compare_with_humans(records,human_val);report['judge_version']=name
        report['response_type']='all'
        validation_reports.append(report)
        for response_type,prefix in [('direct','direct_'),('reasoning','trace_')]:
            subset=[r for r in records if r['response_id'].startswith(prefix)]
            human_subset=human_val[human_val.response_id.str.startswith(prefix)]
            sub_report=compare_with_humans(subset,human_subset)
            sub_report['judge_version']=name;sub_report['response_type']=response_type
            validation_reports.append(sub_report)
    judge_validation_report=pd.concat(validation_reports,ignore_index=True)
    display(judge_validation_report)
    judge_validation_report.to_json(OUTPUT/'judge_validation_report.json',orient='records',indent=2)
    display(pairwise_agreement(human_labels[human_labels.response_id.isin(validation_ids)]))
    # Also inspect final-answer original-metric comparison and process-failure flags for reserved cases.
    display(direct_metric_cases[direct_metric_cases.response_id.isin(validation_ids)])
    display(trace_metric_cases[trace_metric_cases.response_id.isin(validation_ids)])


### TODO — can you trust this LLM judge?

Discuss where its judgments align with human evidence and where they fail. Does it detect correct-answer/wrong-process cases? Can it reject equivalent valid solutions? What do disagreement, missed-error and coverage results reveal? How would judging mistakes change your conclusions? Clearly separate checks performed from proposed future checks.

## Optional GPT judge versus small Qwen judge

If available, compare on the same settled rubric, references, response set and requested output budget. Record model/version, settings, usage, costs and limitations. GPT is a candidate judge, not a replacement for human labels. Otherwise discuss a controlled comparison as a proposal in your personal report.

In [ ]:
# Optional endpoint setup; consult docs/MODEL_GUIDE.md and your group's budget first.
# from helpers.models import EndpointModel
# import os
# from google.colab import userdata
# os.environ['MODEL_API_KEY']=userdata.get('MODEL_API_KEY')
# gpt_judge=EndpointModel('gpt-4o-mini','https://api.openai.com/v1',max_new_tokens=1024,temperature=0)
# gpt_val=run_judges(validation_items,dev_rows,references,gpt_judge,JUDGE_RUBRIC_V2,'gpt_validation',OUTPUT/'judges')
# display(compare_with_humans(gpt_val,human_val))
# TODO: compare evidence, aspect errors and usage on matched responses, not model names alone.

## Errors in intermediate steps

Use the settled judge on matched outputs across solving methods. Separate extraction/time-point, formula/scoring rule, units, arithmetic, precision and formatting. Support each attribution with an excerpt and reference check; distinguish the first error from downstream consequences. Allow multiple error types when supported by evidence. The judge's performance on the sampled direct and reasoning sets does not establish reliability on every later method. Review additional matched cases when needed.

The helper exports a table; **TODO:** choose meaningful summaries and case studies. Report pass rates with unobservable/error coverage, and identify claims supported only by model judging. A direct-only response has no observable process.

In [ ]:
RUN_ERROR_ANALYSIS=False
if RUN_ERROR_ANALYSIS:
    if not JUDGE_RUBRIC_V2:raise ValueError('Settle your judge before method error analysis.')
    flat_outputs=[r for rows in method_outputs.values() for r in rows]
    items=[{'response_id':f"{r['method']}:{r['id']}",'question_id':r['id'],'candidate_raw':r['raw']} for r in flat_outputs]
    method_judgments=run_judges(items,dev_rows,references,judge_model,JUDGE_RUBRIC_V2,'method_errors_dev',OUTPUT/'judges')
    method_error_table=error_table(flat_outputs,dev_rows,method_judgments)
    method_error_table.to_csv(OUTPUT/'method_error_analysis.csv',index=False)
    display(method_error_table.groupby(['method','kind','process']).size().rename('n').reset_index())
    display(method_error_table[(method_error_table.baseline_pass)&(method_error_table.process=='fail')])
    # TODO: evidence checks, first-error/dependency analysis and matched case studies.

## Final test comparison

Freeze solving prompts and the judge before running test questions. Evaluate every method on the same questions. If compute is limited, select a reproducible test subset and report its sampling. These are held-out solving results; judge validation above uses a separate reserved human-review set. Do not describe unreviewed test judgments as human-validated truth.

In [ ]:
RUN_FINAL_TEST=False
if RUN_FINAL_TEST:
    if not RUN_COT or not RUN_ONE_SHOT or not JUDGE_RUBRIC_V2:raise ValueError('Complete CoT, one-shot CoT and judge design first.')
    import random
    TEST_N=32  # Optional: increase to len(test_rows); report the chosen size and sampling.
    final_rows=random.Random(2026).sample(test_rows,TEST_N)
    solvers={'direct':direct_solver,'cot':solve_cot,'one_shot':solve_one_shot}
    if RUN_GROUNDED:solvers['grounded']=solve_grounded
    if RUN_REFINE:solvers['refined']=solve_refined
    prompt_tag=json.dumps({'cot':COT_INSTRUCTION,'one_shot':ONE_SHOT_INSTRUCTION,'grounded':GROUNDED_INSTRUCTION,
                          'critique':CRITIQUE_INSTRUCTION,'revision':REVISION_INSTRUCTION})
    final_outputs={name:run_experiment(final_rows,model,solver,'test_'+name,OUTPUT,tag=prompt_tag)
                   for name,solver in solvers.items()}
    final_scores=pd.concat([score_run(rows,final_rows) for rows in final_outputs.values()],ignore_index=True)
    display(summary(final_scores))
    flat=[r for rows in final_outputs.values() for r in rows]
    items=[{'response_id':f"{r['method']}:{r['id']}",'question_id':r['id'],'candidate_raw':r['raw']} for r in flat]
    final_judgments=run_judges(items,final_rows,references,judge_model,JUDGE_RUBRIC_V2,'test_judgments',OUTPUT/'judges')
    error_table(flat,final_rows,final_judgments).to_csv(OUTPUT/'test_error_analysis.csv',index=False)
    final_scores.to_csv(OUTPUT/'original_metric_test.csv',index=False)

## Group presentation and findings

Present the original evaluation, human rubric and independent agreement, adjudication decisions, judge improvement/validation, and comparative process errors. Explain limitations and show concrete evidence. Submit shared artifacts and contributions. A new solving pipeline or judge training is optional, not required for a strong data-centric study.

### TODO — group interpretation

Add your main findings and presentation outline here, or link your submitted presentation/findings document.



### TODO — write your analysis 

Which evaluation dimensions worked credibly with an LLM judge, and which still require human involvement? How do your results support this boundary? If tested, how did GPT and the small Qwen judge differ; if not, how would you compare them fairly? Could you improve reliability through new judge-training data, independently checked labels, targeted errors, equivalent valid solutions and better splits? What would remain difficult to validate?

Discuss successes, failures, next steps and untested ideas. Do not present a proposal as an executed result. Each student supplies their own reflection, even though the experiments and presentation are collaborative.

## Save the shared work

Download the archive plus the completed notebook. Include all original human labels, adjudication notes, rubric versions, prompts, outputs, judge failures and summaries. Also submit the group presentation/findings and each individual report. Protect independent labels from overwriting; changing the rubric should create a documented version.

In [ ]:
import shutil,tempfile
with tempfile.TemporaryDirectory() as d:
    bundle=Path(d)/'submission';bundle.mkdir()
    shutil.copytree(OUTPUT,bundle/'outputs')
    shutil.copytree('helpers',bundle/'helpers',ignore=shutil.ignore_patterns('__pycache__'))
    shutil.copytree('docs',bundle/'docs')
    for name in ['requirements.txt','PROJECT.pdf']:
        shutil.copy2(name,bundle/name)
    archive=shutil.make_archive(str(Path.cwd()/'project_results'),'zip',bundle)
print('Results archive:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass